In [0]:
02_train_model

In [0]:
import mlflow

import mlflow.spark

from pyspark.ml.feature import StringIndexer

from pyspark.ml.feature import OneHotEncoder

from pyspark.ml.feature import VectorAssembler

from pyspark.ml import Pipeline

from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
df=spark.table(
f"{catalog}.gold.ml_claim_features"
)

In [0]:
categorical=[
"gender",
"race",
"speciality",
"encounter_class"
]

In [0]:
indexers=[
StringIndexer(
inputCol=c,
outputCol=c+"_idx",
handleInvalid="keep"
)
for c in categorical
]

In [0]:
encoders=[
OneHotEncoder(
inputCol=c+"_idx",
outputCol=c+"_vec"
)
for c in categorical
]

In [0]:
assembler=VectorAssembler(

inputCols=[

"patient_age",

"amount",

"payments",

"adjustments",

"outstanding",

"previous_claim_count",

"gender_vec",

"race_vec",

"speciality_vec",

"encounter_class_vec"

],

outputCol="features"
)

In [0]:
rf=RandomForestClassifier(

labelCol="risk_label",

featuresCol="features",

numTrees=100,

maxDepth=8
)

In [0]:
pipeline=Pipeline(

stages=

indexers+

encoders+

[assembler,rf]
)

In [0]:
train,test=df.randomSplit(
[0.8,0.2],
seed=42
)

In [0]:
mlflow.set_experiment(
"/Shared/Healthcare_Claim_Risk"
)

with mlflow.start_run():

    model=pipeline.fit(train)

    predictions=model.transform(test)

    evaluator=BinaryClassificationEvaluator(
        labelCol="risk_label"
    )

    auc=evaluator.evaluate(predictions)

    mlflow.log_metric(
        "auc",
        auc
    )

    mlflow.log_param(
        "numTrees",
        100
    )

    mlflow.spark.log_model(
        model,
        "claim_risk_model"
    )

    print(f"AUC : {auc}")

In [0]:
predictions.write.mode(
"overwrite"
).saveAsTable(
f"{catalog}.gold.claim_predictions"
)